# PromptMap: Kaggle Hugging Face Runner

Run the PromptMap safety evaluation sequentially on Kaggle GPU. Each model is loaded, evaluated, unloaded, and saved in the JSON/CSV/SQLite format used by the PromptMap frontend.

## Kaggle settings

Before running this notebook:

1. Open **Settings** in the Kaggle notebook.
2. Set **Accelerator** to **GPU T4**.
3. Turn **Internet** on.
4. Accept the Hugging Face licenses for Llama and Gemma if you plan to run them.

In [ ]:
!git clone https://github.com/Ankit-git463/Open-Source-LLM-Prompts-Safety-Analysis.git
%cd Open-Source-LLM-Prompts-Safety-Analysis

In [ ]:
!pip install -q transformers accelerate sentencepiece safetensors bitsandbytes huggingface_hub

In [ ]:
!nvidia-smi

## Hugging Face login

Run this cell if you need gated Llama or Gemma models. Create a read-only token at https://huggingface.co/settings/tokens and paste it when prompted.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!ls
!find rules -maxdepth 2 -type f | head -20
!find SystemPrompts -maxdepth 1 -type f

## Optional: run one small model first

This is a smoke run using Qwen and prompt-stealing rules.

In [ ]:
!python scripts/kaggle_hf_runner.py --models qwen2.5 --iterations 1 --rule-type prompt_stealing --load-in-4bit --max-new-tokens 128

## Run all models sequentially

Remove `--rule-type prompt_stealing` to run every rule category. The exact repository IDs are configured inside `scripts/kaggle_hf_runner.py`.

In [ ]:
!python scripts/kaggle_hf_runner.py --models openchat mistral qwen2.5 phi4-mini llama3.2 qwen3.5 gemma3 --iterations 2 --load-in-4bit --max-new-tokens 256

In [ ]:
import json
from pathlib import Path

index_path = Path('results/runs_index.json')
if index_path.exists():
    runs = json.loads(index_path.read_text())
    for run in runs:
        print(f"{run['model']}: {run['pass_rate']}% ({run['passed']}/{run['total']}) - {run['job_id']}")
else:
    print('No runs_index.json found.')

In [ ]:
!find results -maxdepth 1 -type f -printf '%f\n' | sort

## Optional: start the frontend in Kaggle

The saved files are already compatible with the frontend. A Kaggle notebook normally cannot expose Flask directly without a tunnel, so the simplest workflow is to download the results and copy them into the local project.

In [ ]:
!zip -r promptmap-results.zip results

Download `promptmap-results.zip` from the Kaggle output panel. Extract its `results` folder into the local PromptMap project, then run `python app.py` and open `http://localhost:5000`.